### Introduction and Objective

This notebook derives AuroraPay key risk indicators (KRIs) from the cleaned datasets, aligned to the product risk register and risk appetite statements. It calculates incident rates, fraud loss ratios, and issue concentrations by risk area, and exports tidy tables for Power BI dashboards and executive reporting.

In [1]:
import pandas as pd
import numpy as np

pd.options.display.float_format = "{:,.4f}".format

**Step 1 – Load processed datasets**

Load the processed datasets created in the data preparation step.

In [2]:
transactions = pd.read_csv("../data/processed/transactions_clean.csv", parse_dates=["timestamp"])
incidents = pd.read_csv("../data/processed/incidents_clean.csv", parse_dates=["start_time", "end_time"])
risk_register = pd.read_csv("../data/processed/risk_register.csv")
issues = pd.read_csv("../data/processed/issues.csv", parse_dates=["opened_date", "target_resolution_date"])

transactions.shape, incidents.shape, risk_register.shape, issues.shape

((100000, 11), (8, 12), (12, 14), (8, 9))

**Step 2 – Transaction KRIs by product and channel**

Compute basic volume and fraud metrics by product and channel to understand where risk and activity are concentrated.

In [3]:
# Overall transactions and volume
total_txn = len(transactions)
txn_total_volume = transactions["amount"].sum()

print("Total transactions:", total_txn)
print("Total volume (CAD):", txn_total_volume)

# By product
kri_txn_by_product = (
    transactions
    .groupby("product")
    .agg(
        txn_count=("transaction_id", "count"),
        txn_volume=("amount", "sum"),
        fraud_txn=("fraud_flag", "sum")
    )
    .reset_index()
)

kri_txn_by_product["fraud_rate"] = kri_txn_by_product["fraud_txn"] / kri_txn_by_product["txn_count"]
kri_txn_by_product

Total transactions: 100000
Total volume (CAD): 2759295.96


,product,txn_count,txn_volume,fraud_txn,fraud_rate
0,Debit,59798,"1,653,687.2000",31,0.0005
1,e-Transfer,40202,"1,105,608.7600",116,0.0029


By channel:

In [4]:
kri_txn_by_channel = (
    transactions
    .groupby(["product", "channel"])
    .agg(
        txn_count=("transaction_id", "count"),
        txn_volume=("amount", "sum"),
        fraud_txn=("fraud_flag", "sum")
    )
    .reset_index()
)
kri_txn_by_channel["fraud_rate"] = kri_txn_by_channel["fraud_txn"] / kri_txn_by_channel["txn_count"]
kri_txn_by_channel

,product,channel,txn_count,txn_volume,fraud_txn,fraud_rate
0,Debit,Mobile,14883,"413,865.3900",8,0.0005
1,Debit,Online,21053,"577,725.0800",12,0.0006
2,Debit,POS,23862,"662,096.7300",11,0.0005
3,e-Transfer,Mobile,10088,"275,165.0600",22,0.0022
4,e-Transfer,Online,14024,"389,164.1100",39,0.0028
5,e-Transfer,POS,16090,"441,279.5900",55,0.0034


**Step 3 – Incident KRIs by risk and severity**

Calculate incident counts, severities, and rates relative to transaction volumes where possible.

In [5]:
# Map severity to a simple severity index
severity_weight = { "Critical": 4, "High": 3, "Medium": 2, "Low": 1 }
incidents['severity_weight'] = incidents['severity'].map(severity_weight)

# Incidents by risk and severity
incidents_by_risk = (
    incidents.groupby(['risk_id', 'severity'])
    .agg(
        incident_count = ('incident_id', 'count'),
        total_duration_minutes = ('duration_minutes', 'sum')
    )
    .reset_index()
)

# Join Risk category
incidents_by_risk = incidents_by_risk.merge(
    risk_register[['risk_id', 'risk_category']],
    on='risk_id',
    how='left'
)

incidents_by_risk

,risk_id,severity,incident_count,total_duration_minutes,risk_category
0,R1,Critical,1,57.0000,Operational
1,R11,Medium,1,"15,839.9833",Technology & Security
2,R2,High,1,95.0000,Operational
3,R3,High,1,"24,479.9833",Fraud
4,R4,Medium,1,"12,959.9833",Fraud
5,R5,High,1,105.0000,Technology & Security
6,R7,High,1,45.0000,Third-Party
7,R9,High,1,"20,095.0000",Regulatory & Oversight


In [6]:
# Overall Rate
total_incidents = len(incidents)

incidents_per_million_txn = total_incidents / total_txn * 1_000_000
print(f'Incidents per 1M transactions(overall): {incidents_per_million_txn}')

Incidents per 1M transactions(overall): 80.0


In [7]:
incidents['date'] = incidents['start_time'].dt.date

incidents_timeline = (
    incidents.groupby(['date', 'severity'])
    .agg(incident_count = ('incident_id', 'count'))
    .reset_index()
)

**Step 4 – Issues KRIs by risk and category**

Summarize open issues by risk and severity to understand residual risk and remediation pressure.

In [8]:
# Filter to open/in-progress
open_status = ['Open', 'In Progress']
issues_open = issues[issues['status'].isin(open_status)].copy()

issues_by_risk = (
    issues_open.groupby(['risk_id', 'severity'])
    .agg(issue_count = ('issue_id', 'count'))
    .reset_index()
)

issues_by_risk = issues_by_risk.merge(
    risk_register[['risk_id', 'risk_category']],
    on='risk_id',
    how='left'
)

issues_by_risk

,risk_id,severity,issue_count,risk_category
0,R11,High,1,Technology & Security
1,R12,High,1,Operational
2,R3,High,1,Fraud
3,R3,Medium,1,Fraud
4,R5,Critical,1,Technology & Security
5,R6,Medium,1,Technology & Security
6,R7,High,1,Third-Party
7,R9,High,1,Regulatory & Oversight


Aggragate by Risk Category:

In [9]:
issues_by_category = (
    issues_by_risk.groupby(['risk_category','severity'])
    .agg(issue_count = ('issue_count', 'sum'))
    .reset_index()
)

issues_by_category

,risk_category,severity,issue_count
0,Fraud,High,1
1,Fraud,Medium,1
2,Operational,High,1
3,Regulatory & Oversight,High,1
4,Technology & Security,Critical,1
5,Technology & Security,High,1
6,Technology & Security,Medium,1
7,Third-Party,High,1


In [10]:
today = pd.to_datetime('2026-03-31') # Assuming end date for Q1 2026
issues_open['overdue'] = issues_open['target_resolution_date'] < today

overdue_summary = (
    issues_open.groupby(['risk_id', 'overdue'])
    .agg(issue_count = ('issue_id', 'count'))
    .reset_index()
)

overdue_summary

,risk_id,overdue,issue_count
0,R11,False,1
1,R12,False,1
2,R3,False,1
3,R3,True,1
4,R5,False,1
5,R6,False,1
6,R7,False,1
7,R9,False,1


**Step 5 – Build a compact KRI overview**

Combine key metrics into a single table that can be used in dashboards and the executive summary.

In [11]:
kri_overview = kri_txn_by_product.copy()

kri_overview.rename(columns={
    'fraud_txn': 'fraud_txn_count',
}, inplace=True)

kri_overview

,product,txn_count,txn_volume,fraud_txn_count,fraud_rate
0,Debit,59798,"1,653,687.2000",31,0.0005
1,e-Transfer,40202,"1,105,608.7600",116,0.0029


In [12]:
# Incident counts per risk (all severities)
incidents_per_risk = (
    incidents.groupby("risk_id")
    .agg(
        total_incidents=("incident_id", "count"),
        critical_high_incidents=("severity", lambda s: (s.isin(["Critical", "High"])).sum())
    )
    .reset_index()
)

# Open High/Critical issues per risk
high_critical_issues = (
    issues_open[issues_open["severity"].isin(["Critical", "High"])]
    .groupby("risk_id")
    .agg(high_critical_issues=("issue_id", "count"))
    .reset_index()
)

kri_by_risk = (
    risk_register[["risk_id", "risk_category", "risk_name", "risk_appetite_statement"]]
    .merge(incidents_per_risk, on="risk_id", how="left")
    .merge(high_critical_issues, on="risk_id", how="left")
    .fillna({"total_incidents": 0, "critical_high_incidents": 0, "high_critical_issues": 0})
)

kri_by_risk

,risk_id,risk_category,risk_name,risk_appetite_statement,total_incidents,critical_high_incidents,high_critical_issues
0,R1,Operational,Major AuroraPay e-Transfer outage,No more than 1 major AuroraPay e-Transfer outa...,1.0000,1.0000,0.0000
1,R2,Operational,Chronic minor instability impacting payments,No more than 4 minor incidents per month causi...,1.0000,1.0000,0.0000
2,R3,Fraud,Social engineering and scam fraud,Fraud loss should not exceed 5 basis points (0...,1.0000,1.0000,1.0000
3,R4,Fraud,Account takeover enabling unauthorized payments,Zero tolerance for systemic control failures; ...,1.0000,0.0000,0.0000
4,R5,Technology & Security,Excessive privileged access to core AuroraPay ...,No unresolved critical findings on privileged ...,1.0000,1.0000,1.0000
5,R6,Technology & Security,Insufficient security monitoring of payments i...,Critical systems must have centralized logging...,0.0000,0.0000,0.0000
6,R7,Third-Party,Critical vendor outage impacting transaction p...,No more than 2 high-impact critical vendor out...,1.0000,1.0000,1.0000
7,R8,Third-Party,Vendor-managed control weaknesses,Zero tolerance for unremediated critical vendo...,0.0000,0.0000,0.0000
8,R9,Regulatory & Oversight,Failure to meet incident and change notificati...,Zero tolerance for confirmed breaches of mater...,1.0000,1.0000,1.0000
9,R10,Regulatory & Oversight,Inability to demonstrate alignment to risk app...,All material product risk decisions must be do...,0.0000,0.0000,0.0000


Export these summary tables for later analysis:

In [13]:
kri_txn_by_product.to_csv("../data/processed/kri_txn_by_product.csv", index=False)
kri_txn_by_channel.to_csv("../data/processed/kri_txn_by_channel.csv", index=False)

incidents_timeline.to_csv("../data/processed/incidents_timeline.csv", index=False)

issues_by_risk.to_csv("../data/processed/issues_by_risk.csv", index=False)
issues_by_category.to_csv("../data/processed/issues_by_category.csv", index=False)

kri_by_risk.to_csv("../data/processed/kri_by_risk.csv", index=False)

**Step 6 – Analytical observations**

- e‑Transfer processes fewer transactions than Debit but exhibits a higher fraud rate, confirming it as a more risk‑intensive rail.
- Operational incidents are concentrated in [list key risks, e.g., R1/R2/R12], with one critical outage and several high‑severity degradations, resulting in a total of X minutes of disruption.
- High/Critical open issues cluster in privileged access and vendor resilience, indicating that residual risk in these areas remains elevated pending remediation.

These patterns will be used to frame the second‑line assessment and challenge in the executive risk summary.